### Pandas Olympic Assignment

- In this project you will be given data sets and asked to answer questions about them. Feel free to work in groups, we will start this in class. 
    1)  How many medals did each country get in total?
    2)  what are the counts of countries that beat the US when the US obtained silver? 
        -   So like what are the counts of the countries that got gold when the US obtained silver?
    3)  Who is the eldest Athlete to get a medal
    4)  List all the Oldest Athletes who got gold while silver and bronze went to younger athletes
    5)  Compare the medals per person for the 5 names you can think of. 

***

#### 1. How many medals did each country get in total?

In [1]:
import pandas as pd
import numpy as np
import re

results = pd.read_csv('Datasets/results.csv', low_memory=False)
bios = pd.read_csv('Datasets/bios.csv')

print(f"Results shape: {results.shape}")
print(f"Bios shape: {bios.shape}")
print(f"\nResults columns: {list(results.columns)}")
print(f"\nBios columns: {list(bios.columns)}")



Results shape: (308408, 11)
Bios shape: (145500, 16)

Results columns: ['Games', 'Event', 'Team', 'Pos', 'Medal', 'As', 'athlete_id', 'NOC', 'Discipline', 'Nationality', 'Unnamed: 7']

Bios columns: ['Roles', 'Sex', 'Full name', 'Used name', 'Born', 'Died', 'NOC', 'athlete_id', 'Measurements', 'Affiliations', 'Nick/petnames', 'Title(s)', 'Other names', 'Nationality', 'Original name', 'Name order']


In [2]:
# Filter only winning rows 
medal_results = results[results['Medal'].isin(['Gold', 'Silver', 'Bronze'])]

# Count medals per country
medal_counts = medal_results.groupby('NOC')['Medal'].value_counts().unstack(fill_value=0)
medal_counts['Total'] = medal_counts.sum(axis=1)
medal_counts = medal_counts.sort_values('Total', ascending=False)

print("\nTotal Medals per Country (Top 20)\n")
print(medal_counts.head(20))
print(f"\nTotal countries: {len(medal_counts)}")


Total Medals per Country (Top 20)

Medal  Bronze  Gold  Silver  Total
NOC                               
USA      1472  2717    1806   5995
URS       680  1076     724   2480
GER       810   817     772   2399
GBR       720   715     764   2199
FRA       737   589     683   2009
ITA       593   605     566   1764
SWE       549   517     585   1651
CAN       570   555     501   1626
AUS       597   380     475   1452
CHN       348   466     431   1245
RUS       420   434     384   1238
HUN       419   458     344   1221
NOR       350   453     413   1216
NED       450   339     378   1167
JPN       407   358     386   1151
GDR       280   392     323    995
FIN       484   224     281    989
SUI       298   212     265    775
KOR       215   269     279    763
ROU       294   166     209    669

Total countries: 156


---

#### 2. what are the counts of countries that beat the US when the US obtained silver? 
- So like what are the counts of the countries that got gold when the US obtained silver?

In [3]:
# US Silver
us_silver = results[(results['NOC'] == 'USA') & (results['Medal'] == 'Silver')]
us_silver_events = set(zip(us_silver['Games'], us_silver['Event']))

# all Gold medal 
gold_results = results[results['Medal'] == 'Gold']

# Filter only events where US got Silver
mask = gold_results.apply(lambda r: (r['Games'], r['Event']) in us_silver_events, axis=1)
gold_beating_us = gold_results[mask]

print("\nCountries that got Gold when US got Silver\n")
print(f"Total instances: {len(gold_beating_us)}")
print(gold_beating_us)


Countries that got Gold when US got Silver

Total instances: 1935
                       Games                        Event            Team  \
270     1900 Summer Olympics       Doubles, Men (Olympic)  Reggie Doherty   
277     1900 Summer Olympics       Doubles, Men (Olympic)  Laurie Doherty   
850     2000 Summer Olympics    Individual, Men (Olympic)             NaN   
1523    1896 Summer Olympics  1,500 metres, Men (Olympic)             NaN   
2337    1988 Summer Olympics       Singles, Men (Olympic)             NaN   
...                      ...                          ...             ...   
306727  2022 Winter Olympics  Ice Hockey, Women (Olympic)          Canada   
308129  2022 Winter Olympics        Team, Mixed (Olympic)             ROC   
308135  2022 Winter Olympics        Team, Mixed (Olympic)             ROC   
308139  2022 Winter Olympics        Team, Mixed (Olympic)             ROC   
308141  2022 Winter Olympics        Team, Mixed (Olympic)             ROC   

        

In [4]:
# Exclude USA from Gold count
gold_beating_us = gold_beating_us[gold_beating_us['NOC'] != 'USA']
beating_country_counts = gold_beating_us['NOC'].value_counts()

print(f"Unique countries: {len(beating_country_counts)}\n")
print(beating_country_counts.head(15))

Unique countries: 72

NOC
CAN    265
AUS    109
URS    109
JPN     95
GBR     79
GER     77
CHN     55
GDR     47
ITA     46
RUS     45
FRA     41
ROU     37
NED     34
HUN     33
FRG     28
Name: count, dtype: int64


Pandas Way

In [5]:
# Filter US for silver medals
us_silver = results[(results['NOC'] == 'USA') & (results['Medal'] == 'Silver')]

# Filter all gold medals
gold_results = results[results['Medal'] == 'Gold']

# Merge gold results with US silver
gold_beating_us = gold_results.merge(
    us_silver[['Games', 'Event']], 
    on=['Games', 'Event'], 
    how='inner'
)

# Exclude USA and count
beating_country_counts = gold_beating_us[gold_beating_us['NOC'] != 'USA']['NOC'].value_counts()
print(beating_country_counts.head(15))

NOC
CAN    3660
JPN    1126
URS    1011
AUS     599
RUS     341
YUG     338
BRA     292
GBR     250
CHN     242
GER     238
NOR     233
NED     211
HUN     195
ROU     191
GDR     141
Name: count, dtype: int64


---

#### 3. Who is the eldest Athlete to get a medal

In [6]:
# Extract birth year and event year
bios['birth_year'] = bios['Born'].str.extract(r'(\d{4})')[0].astype(float)
medal_results['event_year'] = medal_results['Games'].str.extract(r'(\d{4})')[0].astype(float)

# Merge medal results with bios
merged = medal_results.merge(
    bios[['athlete_id', 'Full name', 'birth_year']],
    on='athlete_id',
    how='inner'
)

# Calculate age and find the row with the max age
merged['age'] = merged['event_year'] - merged['birth_year']
eldest = merged.loc[merged['age'].idxmax()]

print("\nEldest Athlete to Get a Medal\n")
print(f"Name: {eldest['Full name']}")
print(f"Country (NOC): {eldest['NOC']}")
print(f"Medal: {eldest['Medal']}")
print(f"Event: {eldest['Event']}")
print(f"Games: {eldest['Games']}")
print(f"Born: {int(eldest['birth_year'])}")
print(f"Age at event: {eldest['age']} years")


Eldest Athlete to Get a Medal

Name: Oscar Gomer•Swahn
Country (NOC): SWE
Medal: Silver
Event: Running Target, Double Shot, Team, Men (Olympic)
Games: 1920 Summer Olympics
Born: 1847
Age at event: 73.0 years


---

#### 4. List all the Oldest Athletes who got gold while silver and bronze went to younger athletes

In [7]:
oldest_list = []

for (game, event), group in merged.groupby(['Games', 'Event']):
    golds = group[group['Medal'] == 'Gold']
    silvers = group[group['Medal'] == 'Silver']
    bronzes = group[group['Medal'] == 'Bronze']
    
    if not golds.empty and not silvers.empty and not bronzes.empty:
        max_gold = golds['age'].max()
        if max_gold > silvers['age'].max() and max_gold > bronzes['age'].max():
            oldest_gold = golds[golds['age'] == max_gold].iloc[0]
            noc_col = next((col for col in oldest_gold.index if 'NOC' in col), 'NOC')
            
            oldest_list.append({
                'name': oldest_gold['Full name'],
                'age': max_gold,
                'event': event,
                'games': game,
                'noc': oldest_gold[noc_col]
            })

q4_df = pd.DataFrame(oldest_list).sort_values('age', ascending=False)
print(q4_df.head(10))

                               name   age  \
122               Oscar Gomer•Swahn  65.0   
53                  Charles•Jacobus  64.0   
100               Oscar Gomer•Swahn  61.0   
96           Joshua Kearney•Millner  61.0   
1484  Nicholas David "Nick"•Skelton  59.0   
31         Chrétien André•Waydelich  59.0   
206              Johan August•Anker  57.0   
1059     Nikolaus "Klaus"•Balkenhol  57.0   
189                    Allen•Whitty  57.0   
337             Paul Hurlburt•Smart  56.0   

                                                 event                 games  \
122   Running Target, Single Shot, Team, Men (Olympic)  1912 Summer Olympics   
53                              Singles, Men (Olympic)  1904 Summer Olympics   
100         Running Target, Single Shot, Men (Olympic)  1908 Summer Olympics   
96              Free Rifle, 1,000 yards, Men (Olympic)  1908 Summer Olympics   
1484                        Individual, Open (Olympic)  2016 Summer Olympics   
31                  Sing

---

#### 5. Compare the medals per person for the 5 names you can think of.

In [8]:
# total medals per person
athlete_medal_counts = medal_results.groupby('athlete_id').agg(
    Gold=('Medal', lambda x: (x == 'Gold').sum()),
    Silver=('Medal', lambda x: (x == 'Silver').sum()),
    Bronze=('Medal', lambda x: (x == 'Bronze').sum())
).reset_index()
athlete_medal_counts['Total'] = athlete_medal_counts['Gold'] + athlete_medal_counts['Silver'] + athlete_medal_counts['Bronze']
athlete_summary = athlete_medal_counts.merge(
    bios[['athlete_id', 'Full name', 'Used name', 'NOC']],
    on='athlete_id',
    how='left'
)

# by last name
five_athletes = ['Spitz', 'Nurmi', 'Latynina', 'Biles', 'Lewis']

comparison = []
for name_part in five_athletes:
    matches = athlete_summary[athlete_summary['Full name'].str.contains(name_part, case=False, na=False)]
    if matches.empty:
        matches = athlete_summary[athlete_summary['Used name'].str.contains(name_part, case=False, na=False)]
    if not matches.empty:
        best = matches.loc[matches['Total'].idxmax()]
        comparison.append(best)

comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df[['Full name', 'NOC', 'Gold', 'Silver', 'Bronze', 'Total']].sort_values('Total', ascending=False)
comparison_df.columns = ['Athlete', 'NOC', 'Gold', 'Silver', 'Bronze', 'Total']

print(f"\nMedals Comparison for 5 Notable Athletes\n")
print(comparison_df.to_string(index=False))


Medals Comparison for 5 Notable Athletes

                           Athlete           NOC  Gold  Silver  Bronze  Total
Larisa Semyonovna•Latynina (Diry-)  Soviet Union     9       5       4     18
              Paavo Johannes•Nurmi       Finland     9       3       0     12
                 Mark Andrew•Spitz United States     9       1       1     11
    Frederick Carlton "Carl"•Lewis United States     9       1       0     10
     Simone Arianne•Biles (-Owens) United States     4       1       2      7


- Googles Method Below

In [9]:
five_athletes = ['Spitz', 'Nurmi', 'Latynina', 'Biles', 'Lewis']
pattern = '|'.join(five_athletes)
comparison_df = athlete_summary[
    athlete_summary['Full name'].str.contains(pattern, case=False, na=False) |
    athlete_summary['Used name'].str.contains(pattern, case=False, na=False)
]

comparison_df = comparison_df[['Full name', 'NOC', 'Gold', 'Silver', 'Bronze', 'Total']].sort_values('Total', ascending=False)
print(comparison_df.to_string(index=False))

                         Full name           NOC  Gold  Silver  Bronze  Total
Larisa Semyonovna•Latynina (Diry-)  Soviet Union     9       5       4     18
              Paavo Johannes•Nurmi       Finland     9       3       0     12
                 Mark Andrew•Spitz United States     9       1       1     11
    Frederick Carlton "Carl"•Lewis United States     9       1       0     10
     Simone Arianne•Biles (-Owens) United States     4       1       2      7
         Steven Earl "Steve"•Lewis United States     3       1       0      4
                Patrick Lewis•Chan        Canada     1       2       0      3
               Arthur Lewis•Newton United States     1       0       2      3
                      Sabine•Spitz       Germany     1       1       1      3
 Sandra Louise "Sandy"•Allen-Lewis     Australia     0       1       2      3
                Paul Snowden•Lewis     Australia     0       1       1      2
       Hayley Jane•Lewis (-Taylor)     Australia     0       1  